# DermAnnotate — ViT Model Comparison

Compares 3 Vision Transformer architectures on HAM10000 skin lesion classification:
1. **Swin-Tiny** (Microsoft) — 28M params
2. **ViT-Base-16** (Google) — 86M params
3. **DeiT-Small-16** (Meta) — 22M params

**Runtime:** Make sure you select **GPU** runtime: `Runtime → Change runtime type → T4 GPU`

## Step 1: Install dependencies

In [ ]:
!pip install -q transformers torch torchvision huggingface_hub pillow

## Step 2: Upload your trained Swin-Tiny model

Run the cell below, then upload these 3 files from `backend/models/swin_tiny/`:
- `config.json`
- `model.safetensors`
- `preprocessor_config.json`

In [ ]:
import os
from google.colab import files

os.makedirs("models/swin_tiny", exist_ok=True)

print("Upload the 3 files from backend/models/swin_tiny/:")
print("  - config.json")
print("  - model.safetensors")
print("  - preprocessor_config.json")
print()
uploaded = files.upload()

for fname in uploaded:
    dest = f"models/swin_tiny/{fname}"
    os.rename(fname, dest)
    print(f"  → {dest}")

print("\nSwin-Tiny model uploaded!")
print("Files:", os.listdir("models/swin_tiny"))

## Step 3: Download HAM10000 dataset from HuggingFace

In [ ]:
import shutil
from pathlib import Path
from huggingface_hub import hf_hub_download, list_repo_files

DATASET_REPO = "hawking32/ham10000_ttv"
DATA_DIR = Path("dataset/ham10000_full")
TEST_DIR = Path("dataset/ham10000_test")

HAM_CLASSES = ["akiec", "bcc", "bkl", "df", "nv", "mel", "vasc"]

print(f"Listing files in {DATASET_REPO}...")
all_files = list(list_repo_files(DATASET_REPO, repo_type="dataset"))
print(f"Found {len(all_files)} files in repo")

# Download train, val, test splits
for split in ["train", "val", "test"]:
    target_dir = TEST_DIR if split == "test" else DATA_DIR
    split_files = [
        f for f in all_files
        if f.startswith(f"{split}/") and f.endswith(".jpg")
        and f.split("/")[1] in HAM_CLASSES
    ]
    print(f"\n{split}: {len(split_files)} images")

    # For test split, remap paths: test/akiec/img.jpg -> dataset/ham10000_test/akiec/img.jpg
    done = 0
    for repo_path in split_files:
        parts = repo_path.split("/")
        if split == "test":
            dest = target_dir / parts[1] / parts[2]
        else:
            dest = target_dir / repo_path

        if dest.exists():
            done += 1
            continue

        dest.parent.mkdir(parents=True, exist_ok=True)
        try:
            local = hf_hub_download(
                repo_id=DATASET_REPO,
                filename=repo_path,
                repo_type="dataset",
                local_dir="_hf_cache",
            )
            shutil.copy2(local, dest)
            done += 1
            if done % 200 == 0:
                print(f"  {done}/{len(split_files)}...")
        except Exception as e:
            print(f"  SKIP {repo_path}: {e}")

    print(f"  {done}/{len(split_files)} ready")

print("\nDataset download complete!")

## Step 4: Verify GPU and setup

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU! Go to Runtime → Change runtime type → T4 GPU")

# Verify dataset
for d in ["dataset/ham10000_full/train", "dataset/ham10000_full/val", "dataset/ham10000_test"]:
    p = Path(d)
    if p.exists():
        count = sum(1 for _ in p.rglob("*.jpg"))
        print(f"{d}: {count} images")
    else:
        print(f"{d}: MISSING!")

# Verify swin_tiny model
swin_path = Path("models/swin_tiny")
if swin_path.exists() and (swin_path / "model.safetensors").exists():
    print("Swin-Tiny model: OK")
else:
    print("Swin-Tiny model: MISSING — go back to Step 2")

## Step 5: Define training & evaluation code

In [ ]:
import time
import csv
import numpy as np
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForImageClassification

# ── Constants ──
DATA_DIR = Path("dataset/ham10000_full")
TEST_DIR = Path("dataset/ham10000_test")
MODELS_DIR = Path("models")
RESULTS_DIR = Path("results")

HAM_CLASSES = {
    "akiec": (0, "Actinic Keratosis"),
    "bcc":   (1, "Basal Cell Carcinoma"),
    "bkl":   (2, "Benign Keratosis"),
    "df":    (3, "Dermatofibroma"),
    "nv":    (4, "Melanocytic Nevi"),
    "mel":   (5, "Melanoma"),
    "vasc":  (6, "Vascular Lesion"),
}
ID2LABEL = {idx: name for _, (idx, name) in HAM_CLASSES.items()}
LABEL2ID = {name: idx for idx, name in ID2LABEL.items()}
NUM_CLASSES = len(HAM_CLASSES)


@dataclass
class ModelConfig:
    model_id: str
    display_name: str
    short_name: str
    needs_head_replace: bool

MODEL_CONFIGS = [
    ModelConfig("gianlab/swin-tiny-patch4-window7-224-finetuned-skin-cancer", "Swin-Tiny", "swin_tiny", False),
    ModelConfig("google/vit-base-patch16-224", "ViT-Base-16", "vit_base", True),
    ModelConfig("facebook/deit-small-patch16-224", "DeiT-Small-16", "deit_small", True),
]


class SkinDataset(Dataset):
    def __init__(self, root_dir, processor):
        self.processor = processor
        self.samples = []
        for short_code, (label_idx, _) in HAM_CLASSES.items():
            cls_dir = Path(root_dir) / short_code
            if cls_dir.exists():
                for img_path in cls_dir.glob("*.jpg"):
                    self.samples.append((img_path, label_idx))
        if not self.samples:
            raise RuntimeError(f"No images found under {root_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        enc = self.processor(images=img, return_tensors="pt")
        return {"pixel_values": enc["pixel_values"].squeeze(0), "labels": torch.tensor(label, dtype=torch.long)}

    def class_counts(self):
        counts = Counter(label for _, label in self.samples)
        return [counts[i] for i in range(NUM_CLASSES)]


def load_model(config, from_saved=False):
    if from_saved:
        path = str(MODELS_DIR / config.short_name)
        return (
            AutoModelForImageClassification.from_pretrained(path),
            AutoImageProcessor.from_pretrained(path),
        )
    processor = AutoImageProcessor.from_pretrained(config.model_id)
    if config.needs_head_replace:
        model = AutoModelForImageClassification.from_pretrained(
            config.model_id, num_labels=NUM_CLASSES,
            id2label=ID2LABEL, label2id=LABEL2ID,
            ignore_mismatched_sizes=True,
        )
    else:
        model = AutoModelForImageClassification.from_pretrained(config.model_id)
    return model, processor


def train_one_model(config, epochs=5, lr=2e-5, batch_size=32):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n{'='*60}")
    print(f"Training: {config.display_name}")
    print(f"{'='*60}")

    model, processor = load_model(config)
    model.to(device)

    param_count = sum(p.numel() for p in model.parameters())
    print(f"  Params: {param_count:,} | Device: {device}")

    train_ds = SkinDataset(DATA_DIR / "train", processor)
    val_ds = SkinDataset(DATA_DIR / "val", processor)
    print(f"  Train: {len(train_ds)} | Val: {len(val_ds)}")

    # Class weights
    counts = train_ds.class_counts()
    total = sum(counts)
    class_weights = torch.tensor(
        [total / (NUM_CLASSES * c) if c > 0 else 0.0 for c in counts],
        dtype=torch.float32,
    ).to(device)

    # Weighted sampler
    sample_weights = [1.0 / counts[label] for _, label in train_ds.samples]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = CosineAnnealingLR(optimizer, T_max=len(train_loader) * epochs, eta_min=lr / 10)

    best_val_acc = 0.0
    save_dir = MODELS_DIR / config.short_name
    t_start = time.perf_counter()

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        for step, batch in enumerate(train_loader, 1):
            pv = batch["pixel_values"].to(device)
            lb = batch["labels"].to(device)
            optimizer.zero_grad()
            loss = F.cross_entropy(model(pixel_values=pv).logits, lb, weight=class_weights)
            loss.backward()
            optimizer.step()
            scheduler.step()
            train_loss += loss.item()
            if step % 50 == 0:
                print(f"    Epoch {epoch} step {step}/{len(train_loader)}  loss={train_loss/step:.4f}")

        # Validate
        model.eval()
        correct = total_val = 0
        with torch.no_grad():
            for batch in val_loader:
                pv = batch["pixel_values"].to(device)
                lb = batch["labels"].to(device)
                preds = model(pixel_values=pv).logits.argmax(dim=-1)
                correct += (preds == lb).sum().item()
                total_val += lb.size(0)

        val_acc = correct / total_val
        print(f"  Epoch {epoch}/{epochs}  loss={train_loss/len(train_loader):.4f}  val_acc={val_acc:.1%}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            save_dir.mkdir(parents=True, exist_ok=True)
            model.save_pretrained(str(save_dir))
            processor.save_pretrained(str(save_dir))
            print(f"    >>> New best ({val_acc:.1%}) saved to {save_dir}")

    train_time = time.perf_counter() - t_start
    print(f"  Done. Best val: {best_val_acc:.1%} in {train_time:.0f}s")

    del model, optimizer, scheduler
    torch.cuda.empty_cache()

    return {"best_val_acc": best_val_acc, "train_time_s": round(train_time, 1), "param_count": param_count}


def evaluate_one_model(config):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n{'='*60}")
    print(f"Evaluating: {config.display_name}")
    print(f"{'='*60}")

    model_path = MODELS_DIR / config.short_name
    if not model_path.exists():
        print(f"  SKIP: No model at {model_path}")
        return {}

    model, processor = load_model(config, from_saved=True)
    model.to(device)
    model.eval()

    param_count = sum(p.numel() for p in model.parameters())

    test_samples = []
    for short_code, (label_idx, _) in HAM_CLASSES.items():
        cls_dir = TEST_DIR / short_code
        if cls_dir.exists():
            for img_path in sorted(cls_dir.glob("*.jpg")):
                test_samples.append((img_path, label_idx))

    print(f"  Test images: {len(test_samples)}")

    correct_top1 = correct_top3 = 0
    per_class_correct = Counter()
    per_class_total = Counter()
    all_preds, all_labels = [], []
    inference_times = []

    with torch.no_grad():
        for img_path, label in test_samples:
            img = Image.open(img_path).convert("RGB")
            pv = processor(images=img, return_tensors="pt")["pixel_values"].to(device)

            t0 = time.perf_counter()
            logits = model(pixel_values=pv).logits
            inference_times.append((time.perf_counter() - t0) * 1000)

            pred = logits.argmax(dim=-1).item()
            all_preds.append(pred)
            all_labels.append(label)

            if pred == label:
                correct_top1 += 1
                per_class_correct[label] += 1
            per_class_total[label] += 1

            top3 = logits.topk(min(3, NUM_CLASSES), dim=-1).indices[0].tolist()
            if label in top3:
                correct_top3 += 1

    top1_acc = correct_top1 / len(test_samples)
    top3_acc = correct_top3 / len(test_samples)
    avg_ms = np.mean(inference_times)

    print(f"  Top-1: {top1_acc:.1%} | Top-3: {top3_acc:.1%} | Avg infer: {avg_ms:.1f}ms")

    per_class_acc = {}
    for short_code, (idx, name) in HAM_CLASSES.items():
        t = per_class_total[idx]
        c = per_class_correct[idx]
        acc = c / t if t > 0 else 0.0
        per_class_acc[name] = acc
        print(f"    {name:<25} {c:>3}/{t:<3}  {acc:.1%}")

    confusion = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
    for p, l in zip(all_preds, all_labels):
        confusion[l][p] += 1

    del model
    torch.cuda.empty_cache()

    return {
        "top1_acc": top1_acc, "top3_acc": top3_acc,
        "avg_infer_ms": round(avg_ms, 1), "param_count": param_count,
        "per_class_acc": per_class_acc, "confusion": confusion.tolist(),
        "n_test": len(test_samples),
    }

print("All functions defined. Ready to train!")

## Step 6: Train ViT-Base and DeiT-Small

This trains both models sequentially. With a T4 GPU, expect ~10-15 min total.

In [ ]:
train_metrics = {}

# Train ViT-Base (skip swin_tiny — already trained)
vit_config = MODEL_CONFIGS[1]  # ViT-Base-16
train_metrics["vit_base"] = train_one_model(vit_config, epochs=5, batch_size=32)

# Train DeiT-Small
deit_config = MODEL_CONFIGS[2]  # DeiT-Small-16
train_metrics["deit_small"] = train_one_model(deit_config, epochs=5, batch_size=32)

print("\n\nTraining complete!")
for name, m in train_metrics.items():
    print(f"  {name}: val_acc={m['best_val_acc']:.1%}, time={m['train_time_s']}s")

## Step 7: Evaluate ALL 3 models on test set

In [ ]:
eval_results = {}
for config in MODEL_CONFIGS:
    result = evaluate_one_model(config)
    if result:
        eval_results[config.short_name] = result

print(f"\nEvaluated {len(eval_results)}/3 models")

## Step 8: Generate comparison report

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Merge metrics
merged = {}
for config in MODEL_CONFIGS:
    name = config.short_name
    if name not in eval_results:
        continue
    merged[name] = {
        "display_name": config.display_name,
        **eval_results[name],
        "train_time_s": train_metrics.get(name, {}).get("train_time_s", "N/A"),
        "best_val_acc": train_metrics.get(name, {}).get("best_val_acc", "N/A"),
    }

n_test = next(iter(merged.values()))["n_test"]

# Print summary table
print("=" * 65)
print("  DermAnnotate — ViT Model Comparison Report")
print("=" * 65)
print(f"Test set: {n_test} images ({n_test // NUM_CLASSES} per class)\n")

hdr = f"{'Model':<20} {'Params':>8} {'Top-1':>7} {'Top-3':>7} {'Train(s)':>9} {'Infer(ms)':>10}"
print(hdr)
print("-" * len(hdr))
for m in merged.values():
    print(f"{m['display_name']:<20} {m['param_count']/1e6:.1f}M{' ':>3} {m['top1_acc']:.1%}{' ':>2} {m['top3_acc']:.1%}{' ':>2} {str(m['train_time_s']):>9} {m['avg_infer_ms']:>10.1f}")

print()
print("Per-Class Top-1 Accuracy:")
class_names = list(ID2LABEL.values())
model_names = [m["display_name"] for m in merged.values()]
cls_hdr = f"{'Class':<25}" + "".join(f" {n:>15}" for n in model_names)
print(cls_hdr)
print("-" * len(cls_hdr))
for cls_name in class_names:
    row = f"{cls_name:<25}"
    for m in merged.values():
        acc = m.get("per_class_acc", {}).get(cls_name, 0.0)
        row += f" {acc:>14.1%}"
    print(row)

# Best model
print()
best = max(merged.values(), key=lambda m: m["top1_acc"])
print(f"Best overall accuracy: {best['display_name']} ({best['top1_acc']:.1%} top-1)")
best_ratio = max(merged.values(), key=lambda m: m["top1_acc"] / (m["param_count"] / 1e6))
print(f"Best accuracy/size:    {best_ratio['display_name']}")

# Save CSV
csv_path = RESULTS_DIR / "model_comparison.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    header = ["model", "params", "top1_acc", "top3_acc", "train_time_s", "avg_infer_ms"] + [f"class_{c}" for c in class_names]
    writer.writerow(header)
    for m in merged.values():
        row = [m["display_name"], m["param_count"], round(m["top1_acc"], 4), round(m["top3_acc"], 4), m["train_time_s"], m["avg_infer_ms"]]
        row += [round(m.get("per_class_acc", {}).get(c, 0.0), 4) for c in class_names]
        writer.writerow(row)
print(f"\nCSV saved to {csv_path}")

# Save text report
txt_path = RESULTS_DIR / "model_comparison.txt"
# (reuse the printed output)
print(f"Report complete!")

## Step 9: Download trained models & results

Run this to zip and download everything. Then copy the files back to your local repo.

In [ ]:
import shutil

# Zip models and results
shutil.make_archive("vit_base_model", "zip", "models/vit_base")
shutil.make_archive("deit_small_model", "zip", "models/deit_small")
shutil.make_archive("results", "zip", "results")

print("Downloading files...")
files.download("vit_base_model.zip")
files.download("deit_small_model.zip")
files.download("results.zip")
print("\nDone! Unzip these into your local backend/ directory:")
print("  vit_base_model.zip   → backend/models/vit_base/")
print("  deit_small_model.zip → backend/models/deit_small/")
print("  results.zip          → backend/results/")